# Clase 11 — Parte 2: Tutorial práctico de Cursor para Data Science

**Taller de Python para Economistas · Data Science con GenAI**

---

## ¿Qué vamos a hacer hoy?

En la Parte 1 entendieron Git y GitHub y clonaron el repo. Ahora vamos a la parte divertida: **usar un asistente de IA como copiloto** para limpiar datos, hacer exploración y correr un modelo predictivo, **sin escribir casi código a mano**.

La herramienta principal del tutorial es **[Cursor](https://cursor.com)**, pero todo lo que aprendan aplica a **Claude Code**, **Antigravity** o cualquier otro agente moderno — lo que cambia es la UI, no los principios.

### Objetivos de aprendizaje

Al terminar este notebook el estudiante:

1. Sabe instalar y configurar Cursor, y entiende las limitaciones de la free tier.
2. Sabe qué es **prompt engineering** y por qué un prompt ingenuo produce resultados mediocres.
3. Sabe qué es un **harness** y un archivo de contexto (`AGENTS.md` / `CLAUDE.md`) — y cómo usarlos para combatir que el modelo es *stateless*.
4. Sabe aplicar **tres recetas de prompt** (exploración, limpieza, modelado) sobre un dataset real.
5. Sabe activar y usar **Plan Mode** para tareas complejas como el modelado.
6. Puede defender las decisiones de limpieza en términos de *missing data*, *selection bias* y tamaño de muestra.

### Cómo usar este notebook

Este tutorial tiene **dos notebooks hermanos**:

- **`02_Tutorial_Cursor.ipynb` ← este** — es el que lees tú. Aquí vas a:
  - Leer la teoría.
  - Copiar los **prompts-receta** que te damos.
  - Responder en las celdas marcadas **✍️ TU TURNO** con tus propias palabras.

- **los notebooks numerados 03-06** — es el notebook de trabajo. Tú **no** escribes código ahí directamente; le pides a Cursor que lo escriba por ti sección por sección. Está pre-estructurado con headers.

> **Regla clave**: responder las preguntas conceptuales es MÁS IMPORTANTE que tener el código corriendo. El código lo escribe la IA; el criterio lo pones tú.

---


## 1. ¿Qué es Cursor?

**Cursor** es un editor de código. Literalmente es un *fork* (una copia modificada) de **Visual Studio Code**, así que si ya usaban VS Code se van a sentir en casa. La diferencia está en **la integración nativa con modelos de IA**:

- Un chat lateral donde le pedís cosas al agente.
- Autocompletado inteligente (**Tab**).
- Un **agente** que puede leer/editar/crear archivos, correr comandos en terminal y modificar múltiples archivos de una.

### Cursor vs alternativas (abril 2026)

| Herramienta | Tipo | Cuándo usarla |
|---|---|---|
| **Cursor** | Editor con agente integrado | Default para data science de todo tipo. UI gráfica familiar. |
| **Claude Code** | CLI (corre en terminal) | Tareas complejas de varios pasos. Mejor contexto largo. Requiere comodidad con terminal. |
| **Antigravity** (Google) | Editor similar a Cursor | Alternativa gratis cuando se acaba tu cuota de Cursor. |
| **GitHub Copilot en VS Code** | Autocompletado + chat | Si ya vivís dentro de VS Code y no querés cambiar. Menos agéntico que Cursor. |

Un dato importante: los cuatro **hablan el mismo idioma subyacente** (los modelos son Claude, GPT-4/5, Gemini). Lo que aprendés de prompt engineering es 100% transferible.


## 2. Instalación

### 2.1 Descargar e instalar

Vayan a **[cursor.com/download](https://cursor.com/download)**. La página detecta su sistema operativo.

**Mac**:
1. Descarguen el `.dmg`.
2. Arrastren **Cursor.app** a la carpeta `Aplicaciones`.
3. Abran desde Launchpad. La primera vez Mac va a pedir confirmación (aplicación descargada de internet) — acepten.

**Windows**:
1. Descarguen el `.exe`.
2. Ejecuten el instalador. Acepten las opciones por defecto.
3. Al final marquen "Add to PATH" si aparece la opción.

### 2.2 Primera configuración

Al abrir Cursor por primera vez les va a pedir:

1. **Login**: usen su cuenta de GitHub (la que crearon en la Parte 1). Es más rápido que crear otra cuenta.
2. **Keybindings**: elijan **VS Code** si nunca usaron un editor serio, o **Vim / Emacs / etc** si ya son usuarios.
3. **Import settings**: si tenían VS Code, Cursor ofrece importar extensiones y temas. Útil.
4. **Language for AI**: pueden dejar en inglés (los modelos responden mejor en inglés) o español — es solo para la UI.

### 2.3 Abrir el repo de la clase

`File → Open Folder…` → navegan a la carpeta `taller-python-prompt-engineering` que clonaron en la Parte 1.

Deberían ver en el panel izquierdo:

```
taller-python-prompt-engineering/
├── AGENTS_TEMPLATE.md   ← template; NO lo lee Cursor automaticamente
├── README.md
├── notebooks/
│   ├── 01_Intro_Git_GitHub.ipynb
│   ├── 02_Tutorial_Cursor.ipynb    ← este archivo
│   ├── 03_EDA.ipynb                ← Seccion 1: exploracion
│   ├── 04_Limpieza.ipynb           ← Seccion 2: limpieza + merge
│   ├── 05_Modelado.ipynb           ← Seccion 3: modelado (Plan Mode)
│   └── 06_Reporte.ipynb            ← Seccion 4: reporte ejecutivo
├── data/
│   ├── admissions.csv
│   └── bmi.csv
└── outputs/                         ← aqui el agente escribe sus artefactos
```


## 3. Free tier — qué esperar (y qué no)

Cursor tiene una capa gratis (**Hobby**) y varias pagas. Para esta clase **la Hobby alcanza**, pero hay que conocer los límites.

| Límite | Hobby (gratis) | Pro ($20/mes) |
|---|---|---|
| Autocompletado (Tab) | 2,000 / mes | Ilimitado |
| Requests premium (Claude/GPT) | ~50 lentos / mes | $20 en créditos + más |
| Modelos disponibles | Limitado | Todos los frontier |
| Agent Mode | Disponible pero limitado | Completo |

### Estrategia para la clase

1. **Empiecen en Hobby**. Hagan esta clase con la free tier.
2. **Si se les acaba la cuota** durante el taller, **instalen Antigravity** (Google) que es gratis y tiene un agente similar — alternen entre ambos.
3. **Si les gusta la experiencia** y van a seguir usándolo para su tesis, el Pro a USD 20/mes tiene sentido. El equivalente en horas de desarrollo es irrisorio.

> **Tip**: cuando están probando prompts y no saben cuál funciona mejor, no gasten requests premium en cada iteración. Prueben primero con un modelo más rápido (p. ej. `claude-haiku` o `gpt-5-mini`) y cuando tengan el prompt pulido lo mandan con el modelo fuerte.

---


## 4. El prompt ingenuo — lo que NO hay que hacer

Antes de enseñarles prompt engineering, vale la pena **ver qué pasa cuando no lo usan**.

> ⚠️ **Pre-requisito**: asegúrense de que **NO existe** un archivo llamado `AGENTS.md` en la raíz del repo. Solo debe estar `AGENTS_TEMPLATE.md`. Si ven un `AGENTS.md`, bórrenlo temporalmente (o renómbrenlo a `AGENTS_backup.md`) antes de correr el experimento. Esto garantiza que el prompt sea verdaderamente *ingenuo* — sin contexto. Cursor auto-carga únicamente archivos llamados exactamente `AGENTS.md`, no `AGENTS_TEMPLATE.md`.

### El experimento

Abran Cursor en el repo, abran el chat (`Cmd/Ctrl + L`), pongan modo **Agent** y peguen esto:

```
Limpia y analiza estas bases de datos
```

Eso es todo. Sin contexto. Sin objetivo. Sin restricciones.

### ¿Qué va a pasar?

Típicamente el agente va a:

1. Asumir algo sobre *cuáles* bases de datos (capaz escoge solo una de las dos, o junta mal las columnas).
2. Hacer una "limpieza" genérica: `dropna()` todo, castear tipos, renombrar a snake_case, y listo.
3. Producir un `describe()` y un par de gráficos cualquiera — sin saber **qué están tratando de predecir**.
4. NO te dice qué está perdiendo en el camino. ¿Cuántas filas borró? ¿Por qué? ¿Hay sesgo en lo que eliminó?

### ¿Por qué el resultado es mediocre?

Porque el modelo es **stateless**: arranca cada sesión sin saber absolutamente nada de tu proyecto. Si tú no le das contexto, **inventa contexto razonable** — lo cual no es lo mismo que contexto correcto.

> **Lección 0**: la calidad del output de un modelo de IA está dominada por la calidad del input. "Garbage in, garbage out", pero a escala industrial.

### ✍️ TU TURNO 1

Corran ese prompt y miren el resultado. En la celda markdown de abajo contesten:

1. ¿Qué dataset(s) eligió procesar el agente?
2. ¿Hizo alguna inferencia/asunción sobre qué quieres lograr con los datos? ¿Cuál?
3. ¿Cuántas filas eliminó en total? ¿Justificó ese número?
4. ¿Mencionó al `Student_ID` como llave?
5. ¿Hizo algún join? ¿Con qué tipo?


### ✍️ Tus respuestas (edita esta celda)

**1. Dataset(s) procesado(s):**

(tu respuesta aquí)

**2. Asunciones del agente sobre el objetivo:**

(tu respuesta aquí)

**3. Filas eliminadas + justificación:**

(tu respuesta aquí)

**4. ¿Mencionó Student_ID?**

(tu respuesta aquí)

**5. ¿Join? ¿Qué tipo?**

(tu respuesta aquí)

---


## 5. Prompt engineering — los cinco principios que importan

Hay cientos de blogs con "50 prompts mágicos". La verdad es que todo se reduce a **cinco principios**:

### 5.1 Sé específico

❌ "Limpia los datos"
✅ "Elimina filas donde `Age < 0` o `Age > 120`; estos son errores de digitación. Para NAs en `High School Percentage`, imputa con la mediana por ciudad. Justifica ambas decisiones."

La especificidad no es opcional. Si no le dices al modelo qué cuenta como outlier, él se inventará un criterio.

### 5.2 Da contexto

❌ "Analiza este CSV"
✅ "El CSV `data/admissions.csv` son registros de admisión de estudiantes. Queremos predecir `Admission Status` (binario). El CSV tiene datos sucios a propósito (errores de digitación, NAs). Antes de predecir, necesito limpieza razonable."

El modelo no puede "ver" tu intención. Tienes que explicitarla.

### 5.3 Indica el formato de salida

❌ "Hazme un análisis"
✅ "Genera código Python en una celda, con comentarios en español explicando cada paso. Después del código genera una celda markdown con 3 bullets de conclusiones. Guarda las figuras en `outputs/figures/`."

Si no especificas el formato, el modelo usa su default, que rara vez es el tuyo.

### 5.4 Define restricciones (qué NO hacer)

❌ (sin restricciones)
✅ "NO uses scikit-learn para los coeficientes (usa statsmodels). NO modifiques archivos en `data/`. NO cargues el CSV completo a memoria si tiene más de 1M filas — usa `chunksize`."

Las restricciones son tan importantes como las instrucciones positivas.

### 5.5 Itera

El primer prompt casi nunca es el bueno. El flujo real es:

```
prompt → resultado → refinar prompt → resultado → refinar → ...
```

Cuando el agente se equivoca, **corregilo inmediatamente**. Una corrección a tiempo es más barata que reiniciar desde cero.

### Un detalle técnico importante

Los modelos son **stateless**: cada conversación parte de cero. Lo que le dijiste ayer **no lo recuerda**. Las soluciones:

- **`AGENTS.md`** (Cursor, Codex, Antigravity) o **`CLAUDE.md`** (Claude Code): un archivo markdown en la raíz del repo que el agente lee automáticamente al empezar. Es como tatuarle el contexto en el brazo.
- **Archivos intermedios de resumen**: en vez de pasarle TODO el dataset, guardás un resumen en `outputs/eda_summary.md` y referenciás ese archivo en prompts siguientes.
- **Dividir en chats**: no pretendas hacer todo en una sola conversación de 100 mensajes. Divide el trabajo en sesiones cortas y empezá cada una con "lee AGENTS.md y outputs/eda_summary.md".


## 6. El harness: por qué AGENTS.md cambia todo

### 6.1 ¿Qué es un harness?

Un **harness** es todo lo que vive entre el modelo y el mundo real. El modelo genera texto; el harness decide qué pueden tocar esas palabras. En un agente como Cursor, el harness incluye:

- **Context builder**: arma el prompt que se le manda al modelo (system prompt + AGENTS.md + conversación + archivos abiertos).
- **Tool layer**: le da al modelo las herramientas de leer/escribir archivos, correr comandos, etc.
- **Permission layer**: decide qué puede ejecutar sin permiso y qué requiere tu aprobación.
- **Memory**: qué se persiste entre turnos.

Cuando entienden el harness, pueden **aprovecharlo** en vez de pelear contra él.

### 6.2 AGENTS.md — el archivo que Cursor lee solo

Este repo vino con `AGENTS_TEMPLATE.md` (ese nombre NO lo carga Cursor automaticamente). La decision pedagogica fue: primero hiciste el prompt ingenuo sin contexto, asi viste la diferencia. **Ahora vamos a activar el contexto**.

Abran `AGENTS_TEMPLATE.md` en la raiz y leanlo completo. Tómense 3 minutos.

Ese archivo tiene:

- **Qué es el proyecto** (objetivo: predecir admisión).
- **Qué datasets hay** y cuáles son sus columnas clave (sin pegar los datos completos — solo la info que un recién llegado necesitaría).
- **Convenciones** (statsmodels para coeficientes, nada de XGBoost).
- **Estructura de carpetas** (dónde vive cada cosa).
- **Reglas operativas** (data es inmutable, outputs van a outputs/).
- **Flujo esperado** (secciones del notebook 03).

Cuando el estudiante le pega un prompt a Cursor, el modelo ya **sabe todo eso** sin que se lo tenga que repetir. Eso es oro puro.

### 6.3 Activa el contexto: copia el template a `AGENTS.md`

Desde la terminal integrada de Cursor (`Ctrl + \`` en Windows/Linux, `Cmd + \`` en Mac para abrirla), parados en la raíz del repo:

**Mac / Linux / Git Bash**:

```bash
cp AGENTS_TEMPLATE.md AGENTS.md
```

**Windows PowerShell**:

```powershell
Copy-Item AGENTS_TEMPLATE.md AGENTS.md
```

Ahora Cursor va a leer `AGENTS.md` automáticamente cada vez que le mandes un prompt. La diferencia con el prompt ingenuo de la Sección 4 la vas a notar de inmediato: el agente ya "sabe" qué estás tratando de lograr.

> **Por qué gitignoreamos `AGENTS.md`**: porque es TU copia, personal. Podés modificarla (agregar reglas para tu estilo, recordatorios sobre tu tesis, etc.) sin afectar la copia que el curso comparte. `AGENTS_TEMPLATE.md` es el template estable que viaja en el repo; `AGENTS.md` es tu copia viva.

### 6.4 Reglas para un buen AGENTS.md

La investigación empírica muestra que:

- ✅ **Cortos**. Máximo 150-200 líneas. Si crece más, dividí en sub-archivos referenciados.
- ✅ **Progressive disclosure**: no cargues TODO al contexto, enseña cómo **encontrar** las cosas.
- ❌ No dupliques lo que ya está en el código (el agente lo puede leer).

### ✍️ TU TURNO 2

Abran `AGENTS.md` y contesten:

1. ¿Cuál es el objetivo del análisis según AGENTS.md?
2. ¿Qué decisión ya está tomada sobre la columna `measurement_code`?
3. Si el agente quisiera instalar `xgboost`, ¿AGENTS.md se lo permitiría? ¿Dónde dice eso?
4. ¿Qué reglas hay sobre escritura en `data/`?

### ✍️ Tus respuestas

**1. Objetivo del análisis:**

(tu respuesta)

**2. Decisión sobre measurement_code:**

(tu respuesta)

**3. ¿xgboost permitido?:**

(tu respuesta)

**4. Reglas sobre data/:**

(tu respuesta)

---


## 7. Sección 1 — Exploración (EDA)

Ahora sí entramos a los datos. Esta es la Sección 1 del notebook de trabajo (los notebooks numerados 03-06).

### 7.1 ¿Qué es EDA y por qué importa?

*Exploratory Data Analysis* es el paso **antes** de tocar los datos. El objetivo no es limpiar — es **entender**. Si limpiás antes de entender, corrés el riesgo de borrar señal pensando que es ruido.

En un EDA razonable chequeamos, para **cada dataset por separado**:

| Dimensión | Qué buscamos |
|---|---|
| **Shape** | ¿Cuántas filas/columnas tenemos? |
| **Tipos** | ¿Cada columna tiene el dtype que debería? (Age debería ser int, no object) |
| **Missing values** | ¿Qué columnas tienen NAs? ¿Cuántos? ¿Concentrados en filas específicas? |
| **Outliers** | ¿Hay valores imposibles (edad -1) o extremos (ingreso 10000×la media)? |
| **Duplicados** | ¿Filas completamente duplicadas? ¿Duplicados por la llave? |
| **Cardinalidad** | En categóricas: ¿cuántos valores únicos? ¿hay typos (`"male"` vs `"Male"` vs `"M"`)? |
| **Tidy** | ¿Los nombres son consistentes? ¿Cada fila es una observación? ¿Cada columna una variable? |

### 7.2 La receta-prompt

Abajo está la **receta-prompt** completa. Cópienla, péguenla en el chat de Cursor (en modo Agent) y esperen.

> ⚠️ **Antes de ejecutar**, asegúrense de tener abierto `los notebooks 03-06 (segun seccion)` en Cursor. El prompt referencia ese archivo.


```
========================================================================
PROMPT RECETA #1 — EXPLORACION (EDA)
========================================================================

Rol: Eres un analista de datos senior, especialista en EDA (exploratory
data analysis) para economistas.

Contexto: lee AGENTS.md antes de empezar. Lo critico:
  - Predecimos Admission Status combinando admissions.csv (base) con bmi.csv
    (enriquecimiento) via Student_ID.
  - Los archivos en data/ son INMUTABLES.
  - Trabajamos en notebooks/03_EDA.ipynb.

Tarea: Para CADA dataset por separado (admissions primero, luego bmi),
  genera celdas en Seccion 1 que reporten:

  1. Shape, uso de memoria, dtypes
  2. Conteo y % de NAs por columna
  3. Para numericas: describe() + deteccion de outliers via IQR
     (marcar umbrales 1.5*IQR como limites)
  4. Para categoricas: value_counts y cardinalidad (incluye NaN)
  5. Duplicados: (a) filas completamente duplicadas (b) duplicados
     por Student_ID
  6. Validacion de rangos razonables:
     - Age en [10, 50]
     - Admission Test Score y High School Percentage en [0, 100]
     - height_m en [1.0, 2.0]
     - weight_kg en [40, 150]
  7. Muestra 5 filas con al menos un problema detectado

NO unas los datasets. NO hagas limpieza. Solo diagnostico.

Formato de salida:
  - ANTES de cada bloque de codigo, una celda markdown corta que explique
    QUE vas a chequear y POR QUE importa.
  - DESPUES del codigo, otra celda markdown con 2-3 bullets de hallazgos.
  - NO uses emojis. Comentarios en espanol.
  - Al final de la Seccion 1, guarda un resumen en outputs/eda_summary.md
    con: hallazgos clave + preguntas abiertas que hay que decidir antes
    de limpiar.

Restricciones:
  - NO modifiques data/*.csv
  - NO instales librerias nuevas
  - NO unas datasets
  - Si el codigo genera figuras, guardalas en outputs/figures/eda/*.png
    (NO las dejes solo inline)

========================================================================
```


### 7.3 Mientras Cursor trabaja

Cuando el agente termine, **no aceptes ciegamente**. Lee cada celda. Pregúntate:

- ¿El código hace lo que yo haría a mano?
- ¿Las conclusiones en markdown son reales o son el agente "llenando espacio"?
- ¿Hay algo raro que el agente pasó por alto?

Si algo no te gusta, **corregí en el acto**. Ejemplos de correcciones:

- "En la detección de outliers de `Age`, no uses IQR — usa el rango fijo [15, 50] que es el rango razonable para esta cohorte."
- "Falta chequear si hay registros donde `height_m` y `weight_kg` son incompatibles (ej. altura 1.50 con peso 200kg)."

### 7.4 ✍️ TU TURNO 3 — Preguntas a responder después del EDA

Una vez Cursor terminó la Sección 1, contesta estas preguntas con números concretos. Esto es lo que un profesor/jefe te preguntaría si vieras tu primer análisis:


### ✍️ Tus respuestas al EDA

**1. Dimensiones de cada dataset:**
- `admissions.csv`: ___ filas × ___ columnas
- `bmi.csv`: ___ filas × ___ columnas

**2. Columnas con NAs en `admissions` (columna → conteo):**

(tu respuesta)

**3. Outliers detectados en `admissions`** (columna → valor(es) problemático(s)):

(tu respuesta)

**4. Outliers o valores raros en `bmi`:**

(tu respuesta)

**5. ¿Hay duplicados por `Student_ID`?**

(tu respuesta)

**6. Cardinalidad de la columna `Gender` en `admissions`** (¿cuántos valores únicos? ¿hay inconsistencia de mayúsculas?):

(tu respuesta)

**7. ¿Qué es la columna `measurement_code` en bmi? ¿Qué opinas?**

(tu respuesta)

**8. Rango de edad en cada dataset** — ¿hay personas en admissions que NUNCA podrían aparecer en bmi por edad?

(tu respuesta)

**9. Las 3 preguntas más importantes que tendrás que decidir antes de limpiar:**

1. (tu respuesta)
2. (tu respuesta)
3. (tu respuesta)

---


## 8. Sección 2 — Limpieza + Merge

Aquí viene la parte **más difícil** conceptualmente (aunque el código sea corto). Limpiar es un ejercicio de **elegir con qué sesgos querés vivir**, no de "arreglar" los datos.

### 8.1 Los tres caminos frente a un valor problemático

Para cada valor problemático (NA, outlier, dato inválido) tenés exactamente **tres opciones**:

| Opción | Cuándo es razonable | Cuándo NO |
|---|---|---|
| **Eliminar la fila** | Pocos casos, el dato es crítico (la llave primaria), sospecha de mal registro completo | Muchas filas afectadas, el problema es sistemático (sesgo) |
| **Eliminar la columna** | Columna sin relevancia teórica, o >40% NAs sin patrón claro | Columna que tu modelo necesita — estás descartando información |
| **Imputar** | NAs aleatorios (MCAR), columna importante, distribución bien entendida | Muchos NAs, patrón no aleatorio, o variable objetivo — NUNCA imputes Y |

### 8.2 Missing data: los tres tipos

| Tipo | Significado | Ejemplo |
|---|---|---|
| **MCAR** (*Missing Completely At Random*) | El que falte no depende de nada | Encuesta digital falló en X% de los envíos |
| **MAR** (*Missing At Random*) | Falta depende de otras variables observadas | Estudiantes más jóvenes contestan menos (depende de `Age`, que sí tenés) |
| **MNAR** (*Missing Not At Random*) | Falta depende de la variable faltante misma | Los rechazados no reportan ingreso porque les da pena — ingreso falta precisamente para los bajos |

**Regla gruesa**: si es MCAR, imputar con la media/mediana/moda está OK. Si es MAR, podés imputar condicional (media por grupo). Si es MNAR, **no podés arreglarlo sin más datos** — hay que reportar como limitación.

### 8.3 Selection bias — el error más frecuente

Si los datos que borrás **no son aleatorios**, tu modelo queda entrenado sobre una muestra sesgada. Ejemplo hipotético en nuestro caso:

> Si borrás todas las admisiones con `Gender` faltante, y resulta que los formularios incompletos son más comunes entre estudiantes de fuera de la ciudad, estás sub-representando esos estudiantes en el modelo. Si luego el modelo se usa para decidir admisiones, perpetúa el sesgo de representación.

**Antes de hacer un `dropna()`**, preguntate: ¿los que se van tienen algo en común? Si sí, hay sesgo.

### 8.4 Tamaño de muestra — el otro trade-off

Tenemos **157 admisiones** y aproximadamente **85-120 quedarán** tras limpieza razonable + merge. Para un modelo predictivo eso es **muy poco**. Cada fila perdida importa.

Regla de oro: si podés imputar con una decisión defendible, **imputa en vez de borrar** cuando la muestra ya es chica.


### 8.5 ✍️ TU TURNO 4 — Decisiones antes del prompt

Este es el paso más importante del tutorial. **Antes** de pegar el prompt de limpieza, **tú** tenés que decidir qué hacer. La IA no decide — ejecuta tus decisiones.

Llena los blancos de abajo CON JUSTIFICACIÓN. Cada decisión va a ir literalmente en el prompt de la Sección 8.6.


### ✍️ Tus decisiones de limpieza

**DEC_1 — NAs en `admissions`** (varias columnas tienen NAs):

- Mi decisión: ___________________________________________
  (Opciones típicas: drop filas con NA en columnas críticas; drop columnas con >X% NAs;
  imputar numéricas con mediana / categóricas con moda; imputar por grupo.)

- Justificación (¿MCAR/MAR/MNAR? ¿Qué perdemos?):
  ___________________________________________

**DEC_2 — Outliers en `Age`** (hay valores negativos tipo `-1`):

- Mi decisión: ___________________________________________

- Justificación:
  ___________________________________________

**DEC_3 — Outliers en `High School Percentage`** (valores fuera de `[0, 100]`):

- Mi decisión: ___________________________________________

- Justificación:
  ___________________________________________

**DEC_4 — Columna `Name`** (¿la usamos para el modelo?):

- Mi decisión: ___________________________________________

- Justificación:
  ___________________________________________

**DEC_5 — Columna `measurement_code` en `bmi`**:

- Mi decisión: ___________________________________________

- Justificación:
  ___________________________________________

**DEC_6 — Duplicados**:

- Mi decisión: ___________________________________________

- Justificación:
  ___________________________________________

**DEC_7 — Tipo de join**:

- Mi decisión: ___________________________________________
  (Opciones: inner join = solo estudiantes con BMI; left join = mantener los 157,
  muchos van a tener NaN en columnas BMI.)

- Justificación en términos de selection bias:
  ___________________________________________

---


### 8.6 La receta-prompt de limpieza

Ahora sí, con tus decisiones listas, pegá este prompt en Cursor:


```
========================================================================
PROMPT RECETA #2 — LIMPIEZA + MERGE
========================================================================

Rol: Analista de datos senior.

Contexto: lee AGENTS.md y outputs/eda_summary.md.
Trabajamos en notebooks/04_Limpieza.ipynb.

Decisiones ya tomadas (pego abajo exactamente como las escribi):

  DEC_1 NAs admissions: <<<PEGA AQUI TU DECISION DEC_1>>>
  DEC_2 Outliers Age: <<<PEGA AQUI TU DECISION DEC_2>>>
  DEC_3 Outliers High School %: <<<PEGA AQUI TU DECISION DEC_3>>>
  DEC_4 Columna Name: <<<PEGA AQUI TU DECISION DEC_4>>>
  DEC_5 Columna measurement_code: <<<PEGA AQUI TU DECISION DEC_5>>>
  DEC_6 Duplicados: <<<PEGA AQUI TU DECISION DEC_6>>>
  DEC_7 Tipo de join: <<<PEGA AQUI TU DECISION DEC_7>>>

Tarea (ejecutar en Seccion 2 del notebook 03):

  1. Cargar data/admissions.csv y data/bmi.csv como DataFrames crudos.
     NO modifiques los archivos originales.

  2. Aplicar cada decision, una por celda, comentada con
     "# DECISION DEC_X: <frase corta>".

  3. Normalizar nombres de columnas a snake_case en AMBOS datasets:
     - admissions: "Admission Test Score" -> admission_test_score, etc.
     - bmi: "Student_ID" -> student_id (debe matchear el de admissions
       EXACTAMENTE, incluido el case, para que el merge no falle).

  4. Aplicar la logica de DEC_3 (rangos razonables) a TODAS las columnas
     tipo score: admission_test_score y high_school_percentage fuera de
     [0, 100] -> NaN, luego imputar.
     Tambien chequear height_m en [1.0, 2.5] y weight_kg en [20, 300]
     en bmi; valores fuera -> drop fila (son errores claros).

  5. Convertir admission_status a binario 0/1 (Accepted=1, Rejected=0).

  6. Hacer el merge segun DEC_7, usando student_id como llave.
     Tras el merge habra columnas duplicadas (gender, age existen en
     AMBOS datasets). Consolidalas en una sola columna si los valores
     son consistentes, o mantene solo la de admissions si difieren.
     Borra las columnas *_bmi redundantes.

  7. Validaciones al final:
     - Shape antes y despues
     - Cuantas filas se perdieron en cada paso (tabla)
     - N efectivo para modelar
     - Balance de clases en admission_status
     - 5 filas muestra

  8. Guardar:
     - outputs/analysis_clean.parquet (formato principal)
     - outputs/analysis_clean.csv (para inspeccion humana)
     - outputs/cleaning_report.md con:
         * decisiones aplicadas
         * conteo de observaciones perdidas por decision
         * discusion de posible selection bias (1 parrafo)

Formato:
  - Cada celda de codigo precedida por markdown de 1-3 lineas
    explicando QUE se va a hacer y POR QUE.
  - Despues de cada bloque, markdown con bullet points de que cambio.
  - Comentarios en espanol.

Restricciones:
  - NO uses imputacion exotica (MICE, KNN-impute). Media / mediana / moda
    o eliminacion basta.
  - NO modifiques data/*.csv
  - Si detectas un problema no cubierto por mis decisiones, PARATE y
    pregunta. No inventes una decision.

========================================================================
```


### 8.7 ✍️ TU TURNO 5 — Post-limpieza


### ✍️ Preguntas post-limpieza

**1. N efectivo tras limpieza + merge:**

___ filas

**2. Balance de clases (accepted vs rejected):**

Accepted: ___ ( ___%)
Rejected: ___ ( ___%)

**3. ¿Cuántas observaciones perdiste en total?** (desde 157 hasta el N final)

(tu respuesta + breve descomposición)

**4. Selection bias — ¿los que se perdieron tienen algo en común?**
(Ej: ¿son principalmente de una ciudad? ¿de un rango de edad específico? ¿con cierto género?)

(tu respuesta)

**5. Si tuvieras que justificarle a un revisor por qué tu dataset final es confiable, ¿qué dirías?**

(tu respuesta)

---


## 8.8 Problemas comunes que vas a encontrar

Antes de pasar a modelado, estos son los tropezones mas frecuentes que vimos
en ejecuciones reales del tutorial. Son predecibles, asi que mejor los mira de una:

### Pitfall 1: El merge falla por case-sensitivity

`admissions.csv` tiene la columna `Student_ID` (capital S, capital ID). `bmi.csv`
tambien tiene `Student_ID`. Si vos renombras a snake_case solo en admissions
(`student_id`) pero olvidas renombrar en bmi, el merge va a tirar
`KeyError: 'student_id'`.

**Solucion**: renombra en AMBOS, y confirma con `df.columns.tolist()` antes del merge.

### Pitfall 2: Columnas duplicadas despues del merge

Tras el merge vas a ver columnas como `gender_adm`, `gender_bmi`, `age_adm`, `age_bmi`.
Si las dejas tal cual, el modelo despues tiene 8 variables donde deberia tener 4.
Consolidalas en una.

### Pitfall 3: La regla de validacion se aplica inconsistente

En DEC_3 te damos la regla para `High School Percentage`. Pero `Admission Test Score`
TAMBIEN deberia estar en `[0, 100]` y si no lo notas, quedan scores > 100 en tu
data final y el coeficiente queda raro. Moraleja: pensa en **principios** (scores
son porcentajes), no en reglas puntuales.

### Pitfall 4: El agente te dice "hecho" sin mostrar nada

Cursor a veces corre codigo "en silencio" y te dice "done" sin printear resultados.
Pedile explicitamente: *"despues de cada celda muestrame el output (shape, head,
o lo que corresponda)"*.

### Pitfall 5: BMI calculado sale raro

Si al calcular `bmi = weight_kg / height_m**2` ves BMIs de 500, es que la altura
esta en `cm` en vez de `m`. En este dataset esta correcto, pero es LA pregunta
de chequeo cuando calculas BMI en cualquier otro contexto.

### Pitfall 6: El modelo es malo. Y esta bien.

Probablemente tu AUC va a estar en [0.4, 0.6]. Eso NO es un bug -- significa
que con N ~ 100 y los features disponibles, la admision no es predecible bien.
**NO inflees el modelo** agregando features hasta que parezca bueno (overfitting).
Reporta el resultado real y discute limitaciones honestamente. Eso es lo que
se espera de un analista serio.


## 9. Sección 3 — Modelado con **Plan Mode**

### 9.1 ¿Qué es Plan Mode?

Cuando una tarea es compleja (varios pasos, decisiones que dependen unas de otras, múltiples archivos), ejecutar "directo" es peligroso: el agente puede tomar un camino, hacer 10 cambios, y recién al final te das cuenta que el approach inicial no era el correcto.

**Plan Mode** resuelve eso: le pides al agente que **primero te muestre un plan**, lo revisás, lo editás, y **solo cuando lo aprobás se ejecuta**.

### 9.2 Cómo activarlo en Cursor

En el chat de Cursor:

1. Posiciona el cursor en el input del agente.
2. Presiona **`Shift + Tab`**.
3. Verás que el modo cambia a **Plan** (indicador arriba del input).
4. Escribe tu prompt normalmente.
5. El agente va a generar un plan markdown en vez de ejecutar código.
6. Revisás/editás el plan. Cuando estés conforme: click en **Execute** (o escribí "approved").

### 9.3 Cuándo conviene Plan Mode

| Usar Plan Mode | NO usar Plan Mode |
|---|---|
| Tarea compleja con varias sub-decisiones | Edición local de una función |
| Modelado / análisis que involucra varios archivos | "Explícame este código" |
| Refactor que toca múltiples archivos | Preguntas factuales |
| Tareas donde dudas de cómo arrancarlo | Cuando ya tenés el plan en la cabeza |

**Regla práctica**: si explicarle a un colega qué hacer te tomaría >3 frases, probablemente vale la pena Plan Mode.


### 9.4 La receta-prompt de modelado

Esta receta está diseñada **específicamente para Plan Mode**. El agente va a generar un plan, tú lo revisás, lo aprobás, ejecuta.

> ⚠️ Antes de pegar, **activá Plan Mode** (`Shift + Tab`).

Tenés varios blancos que llenar con tus decisiones de modelado:


### ✍️ Decisiones de modelado (llenar antes de prompt)

**M_1 — Predictores que tienen sentido teórico:**

(ej: age, gender, high_school_percentage, admission_test_score, bmi_calculado = weight_kg / height_m²)

Los míos: _______________________

**M_2 — Variables categóricas a convertir a dummies:**

Las mías: _______________________

**M_3 — Variables a estandarizar** (si las hay):

(ej: age y bmi_calculado — útil para comparar coeficientes del logit):

Las mías: _______________________

**M_4 — ¿Estratificar el split train/test por admission_status?**

Mi elección (sí/no): ___________

Justificación: ___________

---


```
========================================================================
PROMPT RECETA #3 — MODELADO (USAR PLAN MODE: Shift+Tab)
========================================================================

Rol: Econometrista aplicado escribiendo codigo pedagogico para
estudiantes de maestria en economia.

Contexto: lee AGENTS.md y outputs/cleaning_report.md.
Trabajamos en notebooks/05_Modelado.ipynb.

Input: outputs/analysis_clean.parquet (dataset limpio y mergeado).

Variable objetivo: admission_status (1 = Accepted, 0 = Rejected).

Mis decisiones de modelado:
  M_1 Predictores: <<<PEGA AQUI>>>
  M_2 Categoricas a dummies: <<<PEGA AQUI>>>
  M_3 Variables a estandarizar: <<<PEGA AQUI>>>
  M_4 Estratificar split: <<<PEGA AQUI>>>

Plan que quiero que elabores (generar primero, EJECUTAR DESPUES de
mi aprobacion):

  1. Cargar outputs/analysis_clean.parquet, mostrar dtypes y shape.

  2. Crear feature "bmi_calculado" = weight_kg / (height_m ** 2)
     si aplica. Comentar que es BMI clasico.

  3. Seleccionar predictores listados en M_1, dropear filas con NA
     en cualquiera de ellos (y reportar cuantas). Variable Y:
     admission_status.

  4. Analisis de correlacion (heatmap) entre predictores numericos.
     Guardar en outputs/figures/model/correlation_heatmap.png.
     Si algun par tiene |r| > 0.8, reportar como alerta de
     multicolinealidad.

  5. Si son >= 3 predictores numericos, calcular VIF
     (statsmodels.stats.outliers_influence.variance_inflation_factor).
     Reportar VIF por variable. Flaggear si VIF > 10.

  6. One-hot encoding de M_2 con drop_first=True.

  7. Estandarizar M_3 (zscore).

  8. Split train/test 70/30 con random_state=42. Estratificar segun
     M_4.

  9. Modelo A: Linear Probability Model usando statsmodels.OLS
     sobre admission_status. Mostrar summary().

 10. Modelo B: Logit usando statsmodels.Logit. Mostrar summary().
     Calcular tambien odds ratios (exp(params)).

 11. Evaluar AMBOS modelos en test:
     - accuracy
     - precision, recall, F1
     - AUC
     - tabla comparativa en markdown

 12. Matriz de confusion del logit sobre test. Guardar figura en
     outputs/figures/model/confusion_matrix.png.

 13. Curva ROC del logit sobre test. Guardar figura en
     outputs/figures/model/roc_curve.png.

 14. Interpretacion economica: elegir 2-3 coeficientes significativos
     y explicar en 2-3 frases cada uno, en lenguaje para maestria en
     economia (efecto marginal, odds ratio, etc).

 15. Discusion honesta de limitaciones (N chico, posibles sesgos,
     overfitting). Minimo 1 parrafo.

 16. Resumen final en outputs/model_report.md con todo lo anterior.

Restricciones:
  - statsmodels para los coeficientes (NO sklearn.linear_model).
  - sklearn.metrics SI para metricas de clasificacion.
  - Semilla fija random_state=42 en todos los lugares relevantes.
  - NO pruebes modelos no pedidos (nada de RF, XGBoost, SVM).
  - Si el logit no converge con los predictores dados, REPORTA
    el warning honestamente y sugeri ajustes. No escondas el problema.
  - Si algun N de grupo en el split queda < 10, avisame.

Formato:
  - Markdown antes de cada celda explicando POR QUE ese paso.
  - Comentarios en espanol.
  - Tablas en markdown, no screenshots.

========================================================================
```


### 9.4.1 Interpretando resultados "malos" honestamente

Cuando Cursor termine de ejecutar el modelo, **probablemente tus metricas te van
a desilusionar**: con N ~ 80-110 y los features que tenemos, es esperable que la
AUC quede en el rango [0.4, 0.6] y que NINGUN coeficiente sea significativo al
95%.

Tu primer impulso va a ser "hay que mejorar el modelo". **Resiste**. Razones:

1. **Overfitting facil**: agregar features hasta que el train score suba es
   exactamente como se construyen modelos malos. Con N=80, 5-6 predictores ya
   es el techo razonable.
2. **No todo se puede predecir**: que un estudiante sea admitido depende de
   muchisimas cosas que no estan en estos datos (entrevista, portafolio,
   ensayo, cuotas, etc). Un R^2 bajo puede ser **la verdad**.
3. **Honestidad > estetica**: un reporte que dice "el modelo predice con
   AUC=0.52 y no encontramos relacion significativa entre BMI y admision" es
   un buen reporte. Uno que dice "tenemos un modelo con accuracy=0.89" sin
   mencionar que train=test es un mal reporte.

Lo que SI podes hacer:
- Reportar intervalos de confianza amplios como son.
- Discutir que con mas N o mejores features (GPA, SAT, etc) el modelo probablemente
  funcionaria mejor.
- Decir que el resultado actual NO soporta usar BMI como criterio de admision
  (lo cual ademas seria problematico eticamente).


### 9.5 ✍️ TU TURNO 6 — Preguntas de interpretación

Cuando Cursor termine el modelado, contestá estas preguntas. Estas son **las que importan** — el código es el medio, la interpretación es el fin.


### ✍️ Preguntas de interpretación del modelo

**1. ¿Cuáles fueron los 3 predictores más significativos (menor p-valor) en el logit?**

(tu respuesta, con el signo del coeficiente y su magnitud)

**2. Interpretá el coeficiente del predictor más importante en el LPM:**

"Un aumento de 1 unidad en _____ se asocia con un cambio de ___ puntos porcentuales en la probabilidad de ser admitido."

**3. Interpretá el mismo predictor en el logit (vía odds ratio):**

"Un aumento de 1 unidad en _____ multiplica las odds de admisión por ___."

**4. ¿Cuál modelo elegirías para comunicar a un decano? ¿Por qué?** (LPM es más fácil de explicar; logit es más correcto estadísticamente):

(tu respuesta)

**5. AUC obtenida:** _____
   ¿Es buena para N ≈ 85-100? ¿Sospechás overfitting?

(tu respuesta)

**6. ¿Qué pasaría si hubieras tenido N = 1,000 en vez de ~100?**
(Pensá en varianza de coeficientes, significancia, capacidad de incluir más predictores)

(tu respuesta)

**7. El modelo dice que alguien con `bmi = X` y `admission_test_score = Y` tiene probabilidad Z de ser admitido. ¿Estás cómodo usando ese modelo para decidir admisiones reales? ¿Por qué sí o por qué no?**

(tu respuesta — esta es LA pregunta ética del curso)

---


## 10. Sección 4 — Presentación de resultados

Trabajas en `notebooks/06_Reporte.ipynb`.

El último paso es **armar un reporte** que alguien que no corrió el código pueda leer. Un buen reporte tiene:

- Resumen ejecutivo (1 párrafo)
- Metodología (cómo limpiaste, qué decidiste)
- Resultados (tablas + 2-3 figuras)
- Limitaciones (honestas)
- Recomendaciones

### Prompt rápido para generar el reporte

```
Genera un reporte final en outputs/final_report.md que integre:
  - Hallazgos clave del EDA (de outputs/eda_summary.md)
  - Decisiones de limpieza (de outputs/cleaning_report.md)
  - Resultados del modelado (de outputs/model_report.md)

Publico: decano de economia que entiende OLS/logit pero no pandas.

Estructura:
  1. Resumen ejecutivo (120 palabras max)
  2. Metodologia (1-2 parrafos)
  3. Resultados principales (tablas + figuras con caption)
  4. Limitaciones (enumerar 3-5 honestamente)
  5. Recomendaciones (3 bullets concretos)

Insertar las figuras ya guardadas con links relativos tipo
![Caption](figures/model/confusion_matrix.png)

Tono: profesional, conciso, sin exageraciones. Si el modelo no es
bueno, decirlo. Si N es chico, decirlo.
```

Pegá eso en Cursor (sin Plan Mode, es tarea simple) y al minuto tenés el reporte.


## 11. Cuando se acaba tu cuota gratis

Si durante el taller ven que Cursor empieza a demorar (queue lento) o les avisa que se agotaron los requests, **no paniqueen**. Hay dos caminos:

### Opción A — Descargar **Antigravity** (Google)

Antigravity es la contraparte gratis de Google. **Gratis, sin cuota estricta en 2026**, y tiene un agente comparable a Cursor.

1. Descarguen de **[antigravity.google.com](https://antigravity.google.com)**.
2. Abran la misma carpeta del repo.
3. El archivo `AGENTS.md` también funciona en Antigravity (por eso es estándar cross-tool).
4. Pueden pegar exactamente los mismos prompts-receta.

### Opción B — Alternar entre Cursor y Antigravity

Algunos de nosotros usamos ambos: Cursor para autocompletado rápido (Tab), Antigravity para tareas agénticas largas. Mientras se regenera tu cuota en uno, trabajás en el otro.

### Opción C — Claude Code (CLI)

Si les gusta la terminal, **Claude Code** es una alternativa más poderosa para tareas largas. Tiene su propio file llamado **`CLAUDE.md`** (misma idea que AGENTS.md). Pueden copiar el contenido de AGENTS.md tal cual a un CLAUDE.md y funciona.

### Cuándo pagar

Cuando noten que:

- Usan el agente > 2h/día.
- Los requests lentos de la free tier les rompen el flow.
- Trabajan en proyectos de su tesis / trabajo, no solo clase.

Ahí el Pro a USD 20/mes es la mejor inversión de ese dinero que van a hacer este año.

---


## 12. Trucos supremos — el resumen que guardás en Google Keep

### 12.1 Atajos de teclado de Cursor (Mac → Windows/Linux)

| Acción | Mac | Windows/Linux |
|---|---|---|
| Abrir chat | `Cmd+L` | `Ctrl+L` |
| Abrir composer (multi-archivo) | `Cmd+I` | `Ctrl+I` |
| Activar Plan Mode | `Shift+Tab` en input | Igual |
| Aceptar suggestion (Tab) | `Tab` | `Tab` |
| Rechazar suggestion | `Esc` | `Esc` |
| Generar código inline | `Cmd+K` | `Ctrl+K` |
| @ para referenciar archivo | `@` en chat | `@` |
| @ para buscar en docs | `@Docs` en chat | `@Docs` |

### 12.2 Buenas prácticas (cheatsheet mental)

1. **Empezá cada sesión con**: "Lee AGENTS.md antes de arrancar."
2. **Si algo no le sale**: no repitas el prompt 5 veces — **aclará el contexto** (dale un ejemplo, mostrale el output esperado).
3. **No des un chat a la muerte**: cuando pasas de ~30 mensajes, el contexto se vuelve ruidoso. Guardá resultados en archivos markdown y arrancá chat nuevo.
4. **Dividí en archivos pequeños**: un archivo = una responsabilidad. `eda_summary.md`, `cleaning_report.md`, `model_report.md`, `final_report.md`. No un monolito de 5000 líneas.
5. **Outputs intermedios en `outputs/`**: el agente puede leerlos en la próxima sesión sin que le repitas todo.
6. **`@archivo.py` en el prompt**: le das al agente acceso directo a ese archivo como contexto.
7. **Cuando dudes, usá Plan Mode**. 10 segundos de plan ahorran 2 horas de rollback.
8. **Verificá el código que no escribiste**. El agente puede estar equivocado con confianza perfecta.

### 12.3 Banderas rojas: cuándo frenar al agente

- Está modificando archivos en `data/` → ¡PARÁ!
- Instaló librerías sin preguntar → revisá si eran necesarias.
- El código corre pero el `describe()` tiene números sospechosos → verificá manualmente.
- El agente dice "I've successfully completed..." pero no mostró los resultados → pedile que los muestre.
- Está "arreglando" un bug escribiendo cada vez más código en vez de entender la causa → resetea, explicale el síntoma, pedile que diagnostique primero.

---


## 13. ¿Qué aprendieron?

Si llegaron hasta acá y llenaron las respuestas honestamente, ahora saben:

- Instalar y configurar Cursor, entender su free tier.
- Distinguir un prompt ingenuo de un prompt bueno (especificidad + contexto + formato + restricciones).
- Qué es un harness y por qué `AGENTS.md` es su mejor amigo.
- Hacer EDA estructurado pidiéndoselo con una receta.
- Tomar decisiones de limpieza defendibles, no al azar.
- Usar **Plan Mode** para tareas complejas como modelado.
- Correr LPM + logit para predecir admisión, e interpretar los resultados.
- Alternar con Antigravity cuando se acaba la cuota.

### Para la próxima

- Probar esto mismo con datos de tu tesis o trabajo.
- Leer los docs oficiales: [cursor.com/docs](https://cursor.com/docs), [claude.com/code](https://docs.claude.com/).
- Construir tu propio `AGENTS.md` para cada proyecto serio.
- Experimentar con otros modelos (Gemini, o3) — a veces uno funciona mejor que otro para tu caso.

---

**Recursos útiles:**

- [Cursor Docs](https://cursor.com/docs)
- [Cursor Plan Mode](https://cursor.com/docs/agent/plan-mode)
- [Best practices for coding with agents (Cursor blog)](https://cursor.com/blog/agent-best-practices)
- [AGENTS.md spec](https://agents.md)
- [Claude Code Best Practices](https://code.claude.com/docs/en/best-practices)
- [Antigravity](https://antigravity.google.com)

---

*Fin del notebook tutorial. Nos vemos en la entrega final.*
